<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/apps/Gemma_4_HDP_Agentic_Security/Gemma_4_HDP_Agentic_Security.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

# 使用 HDP 保護 Gemma 4 代理程式工作流程

**作者：** 機密，Helixar Limited ([@confidential](https://github.com/asiridalugoda)) | [helixar.ai](https://helixar.ai)

## 開始之前

此notebook需要 GPUruntime。若要在Colab中啟用 GPU：1. 轉到**執行時→更改runtime類型**
2. 將**硬體加速器**設定為**GPU**（T4足以用於 E4B）
3. 點選**儲存**

您還需要 **Hugging Face token** 下載 Gemma 4（門控模型）：1. 前往 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. 建立具有 **讀取** 存取權限的 token
3. 接受 Gemma 4 模型授權：[huggingface.co/google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it)
4. 執行下面的 cell 進行身份驗證

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# 使用 HDP 保護 Gemma 4 代理程式工作流程

**人類委託來源 (HDP)** 是一種開放協議，為 AI agent 函數呼叫添加加密監管鏈 — 確保每個工具呼叫都可以追溯到授權的人類主體。
此 notebook 示範如何將 HDP 與 Gemma 4 的本機函數呼叫功能整合以：
- **驗證** Gemma 4 的函數呼叫在執行前已得到人類主體的授權
- **依不可逆性將動作分類（唯讀 → 不可逆 → 物理驅動）
- **阻止**中間件層未經授權或超出範圍的工具調用
- **使用預執行日誌審核**每個決定

這對於邊緣裝置（Jetson Nano、Raspberry Pi）上的Gemma 4 部署尤其重要，其中模型可能會在沒有外部授權檢查的情況下引導實體執行器離線。
**參考：**- HDP IETF 草稿：[draft-helixar-hdp-agentic-delegation-00](https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/)
- HDP-P（物理 AI agents）：[DOI 10.5281/ZENODO.19332440](https://doi.org/10.5281/ZENODO.19332440)
- Helixar：[helixar.ai](https://helixar.ai)

## 設定

In [ ]:
!pip install -q transformers torch cryptography

In [ ]:
# Download the middleware
!wget -q https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/Gemma_4_HDP_Agentic_Security/hdp_middleware.py

from hdp_middleware import (
    HDPDelegationToken,
    HDPMiddleware,
    IrreversibilityClass,
    DEFAULT_TOOL_CLASS_MAP,
)
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
import json

## 1. 載入 Gemma 4

我們在此示範中使用 4B 有效模型。對於生產 agentic 部署，建議使用 26B MoE 或 31B Dense 型號。

In [ ]:
from transformers import pipeline

# For edge/robotics use cases: swap to google/gemma-4-E2B-it
MODEL_ID = "google/gemma-4-E4B-it"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",
)

## 2. 定義工具

Gemma 4 使用結構化 JSON 函數呼叫。我們定義了一個涵蓋不同 IrreversibilityClasses 的工具集來示範中間件的分類行為。

In [ ]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"}
            },
            "required": ["location"]
        }
    },
    {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"]
        }
    },
    {
        "name": "delete_file",
        "description": "Permanently delete a file by path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "actuate_robot_arm",
        "description": "Command a robot arm to move to a target position.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint_angles": {"type": "array", "items": {"type": "number"}},
                "force_limit_n": {"type": "number"}
            },
            "required": ["joint_angles"]
        }
    }
]

# Tools indexed by name for lookup
TOOL_REGISTRY = {t["name"]: t for t in TOOLS}
print(f"Registered {len(TOOLS)} tools")

## 3. 發行 HDP 委託代幣

人類主體產生 Ed25519 金鑰對並發出 HDT，其中指定：- 允許agent調用哪些工具
- agent 可以作用的最大不可逆性等級
- token 的生命週期

In [ ]:
# Human principal generates their signing keypair
# In production: loaded from secure key storage (HSM, OS keychain, etc.)
principal_private_key = Ed25519PrivateKey.generate()
principal_public_key = principal_private_key.public_key()

# Issue an HDT authorizing the Gemma 4 agent to call weather queries
# and send emails (Class 0 and Class 2), but NOT delete files or actuate hardware
token = HDPDelegationToken.issue(
    principal_id="alice@example.com",
    agent_id="gemma4-agent-01",
    scope=["get_weather", "send_email"],
    max_class=IrreversibilityClass.CLASS_2,
    ttl_seconds=3600,
    private_key=principal_private_key,
)

print(json.dumps(token.to_dict(), indent=2))

## 4.初始化 HDP 中介軟體

中間件僅取得主體的**公鑰**－它進行驗證但不能發出tokens。

In [ ]:
audit_log = []

# Confirmation callback for Class 2 (irreversible) actions.
# In production: this would invoke a push notification, SMS OTP,
# or hardware confirmation device to the human principal.
def require_human_confirmation(tool_name: str, parameters: dict) -> bool:
    print(f"\n⚠️  Class 2 action requested: {tool_name}")
    print(f"   Parameters: {json.dumps(parameters, indent=4)}")
    response = input("   Confirm? [y/N]: ").strip().lower()
    return response == "y"

middleware = HDPMiddleware(
    public_key=principal_public_key,
    tool_class_map=DEFAULT_TOOL_CLASS_MAP,
    confirmation_callback=require_human_confirmation,
    audit_log=audit_log,
)

print("HDP middleware initialised.")

## 5. Gemma 4 函數呼叫 → HDP Gate → 工具執行

這是核心整合模式。 Gemma 4 產生的每個函數呼叫在轉送到工具執行之前都會透過 `middleware.gate()`。

In [ ]:
# Simulated Gemma 4 function call outputs
# In production these come from parsing Gemma 4's structured JSON output
gemma_function_calls = [
    # ✅ Should ALLOW — Class 0, in scope
    {"name": "get_weather", "parameters": {"location": "Auckland"}},

    # ⚠️  Should CONFIRM then ALLOW — Class 2, in scope
    {"name": "send_email", "parameters": {
        "to": "bob@example.com",
        "subject": "Weekly report",
        "body": "Please find attached."
    }},

    # ❌ Should BLOCK — Class 2, NOT in HDT scope
    {"name": "delete_file", "parameters": {"path": "/data/important.csv"}},

    # ❌ Should BLOCK — Class 3, physical actuation
    {"name": "actuate_robot_arm", "parameters": {
        "joint_angles": [0.0, -1.57, 0.0, -1.57, 0.0, 0.0],
        "force_limit_n": 50.0
    }},
]

print("=" * 60)
print("HDP VERIFICATION RESULTS")
print("=" * 60)

for call in gemma_function_calls:
    result = middleware.gate(call, token)

## 6. 審核日誌

每個決定在執行前都會被記錄下來。這就是 HDP 審計追蹤——以加密方式連結記錄授權內容、授權者以及授權時間。

In [ ]:
print("\nAUDIT LOG")
print("-" * 60)
for i, entry in enumerate(audit_log):
    status = "✅ ALLOWED" if entry.allowed else "❌ BLOCKED"
    print(f"{i+1}. {status} | {entry.tool_name} | {entry.action_class.name} | {entry.reason}")

## 7. token 過期和範圍違規演示

證明無論操作類別如何，過期的 tokens 和超出範圍的呼叫都會被阻止。

In [ ]:
import time

# Issue a token that's already expired
expired_token = HDPDelegationToken.issue(
    principal_id="alice@example.com",
    agent_id="gemma4-agent-01",
    scope=["get_weather"],
    max_class=IrreversibilityClass.CLASS_0,
    ttl_seconds=-1,   # expired immediately
    private_key=principal_private_key,
)

print("Testing expired token:")
middleware.gate({"name": "get_weather", "parameters": {"location": "Auckland"}}, expired_token)

print("\nTesting call outside HDT scope:")
middleware.gate({"name": "delete_file", "parameters": {"path": "/etc/passwd"}}, token)

## 8. 邊緣/機器人部署（HDP-P）

對於在 Jetson Nano 或 Raspberry Pi 上執行並指導實體致動器的 Gemma 4 E2B/E4B，請使用 HDP-P extension。關鍵的補充是：
- **實作上下文** — 將 token 綁定到特定的硬體 ID
- **策略證明** — 將部署的模型權重雜湊到token
- **車隊授權限制** — 防止機器人車隊橫向移動
- **執行前日誌記錄** — 發出執行器指令*之前*寫入審核記錄

有關完整的 EDT extension 結構，請參閱 [HDP-P 規格](https://doi.org/10.5281/ZENODO.19332440)。

In [ ]:
# Minimal HDP-P Embodied Delegation Token (EDT) extension example
# This shows how to attach physical constraints to an HDT

hdp_p_extension = {
    "hdp-p": {
        "version": "0.1",
        "embodiment": {
            "type": "mobile",
            "platform": "raspberry-pi-5",
            "hardware_id": "rpi-serial-XXXX",   # TPM-attested in production
            "workspace": "lab-zone-a"
        },
        "action_scope": {
            "permitted_actions": ["move_base", "read_sensor"],
            "excluded_zones": ["human-workspace"],
            "force_limit_n": 10.0,
            "max_velocity_ms": 0.5
        },
        "irreversibility": {
            "max_class": 1,                       # Class 1 max for this token
            "class2_requires_confirmation": True,
            "class3_prohibited": True
        },
        "policy_attestation": {
            "policy_hash": "sha256:abc123...",    # SHA-256 of deployed model weights
            "training_run_id": "gemma4-e2b-it",
            "sim_validated": True
        },
        "delegation_scope": {
            "fleet_delegation_permitted": False,   # No lateral movement
            "max_delegation_depth": 0
        }
    }
}

print("HDP-P EDT extension structure:")
print(json.dumps(hdp_p_extension, indent=2))

## 概括

| 層 | 它解決什麼問題 | 工具 ||---|---|---|
| Gemma 4個函數調用 | 模型生成結構化工具調用 | `pipeline("text-generation")` || HDP 中介軟體 | 這通電話是經過人授權的嗎？ | `HDPMiddleware.gate()` || HDP-P EDTextension | 此身體動作是否在授權範圍內？ | `hdp_p_extension` || 審核日誌 | 每個決定執行前的記錄 | `audit_log` |
完整的 HDP 規範（IETF 草案）、HDP-P 配套論文、TypeScript SDK 和 Python 綁定可在以下位置取得：
- **IETF 草案：** https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/
- **HDP-P 紙張：** https://doi.org/10.5281/ZENODO.19332440
- **GitHub:** https://github.com/Helixar-AI
- **站點：** https://helixar.ai